In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn import linear_model
from sklearn import model_selection
# from sklearn import metrics
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    r2_score,
    classification_report
)
from sklearn import preprocessing
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from gensim.models import KeyedVectors
# from gensim.models.fasttext import load_facebook_model
import gensim.downloader as api
import re
from catboost import CatBoostClassifier
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import GridSearchCV
from tqdm import tqdm


TEST_SIZE = 0.2
RANDOM_SEED = 42

In [13]:
tokens = pd.read_csv('../data/comments_parsed.csv')

In [22]:
comments = pd.read_csv("../data/toxic_comments_with_morph.csv").drop(columns='Unnamed: 0')
comments = comments.drop(columns=['cleaned_comment', 'comment_without_punct', 'swearing.1'])

In [29]:
df = pd.concat([comments, tokens], axis=1)

In [30]:
df

,label,comment,length_sym,length_words,av_word_len,is_normal,swearing,has_positive_emoji,has_negative_emoji,has_obscene_emoji,imperative,2nd_prsn_count,3nd_prsn_count,adj_count,vocative,expr_punct,tokens,lemma_pos,feats
0,INSULT,скотина! что сказать,20,3,5.666667,False,False,False,False,False,False,0,0,0,True,False,скотина\tсказать,скотина_NOUN\tсказать_VERB,Animacy=Inan|Case=Nom|Gender=Fem|Number=Sing\t...
1,NORMAL,я сегодня проезжала по рабочей и между домами ...,180,28,5.214286,True,False,False,False,False,False,0,0,3,False,False,сегодня\tпроезжала\tрабочей\tдомами\tснитенко\...,сегодня_ADV\tпроезжать_VERB\tрабочий_ADJ\tдом_...,Degree=Pos\tAspect=Perf|Gender=Fem|Mood=Ind|Nu...
2,NORMAL,очередной лохотрон. зачем придумывать очередно...,379,54,5.777778,True,False,False,False,False,False,0,0,8,False,True,очередной\tлохотрон\tпридумывать\tочередной\tн...,очередной_ADJ\tлохотрон_NOUN\tпридумывать_VERB...,Animacy=Inan|Case=Acc|Degree=Pos|Gender=Masc|N...
3,NORMAL,"ретро дежавю ... сложно понять чужое сердце , ...",72,10,5.700000,True,False,False,False,False,False,0,0,0,False,True,ретро\tдежавю\tсложно\tпонять\tчужое\tсердце\t...,ретро_NOUN\tдежавю_NOUN\tсложный_ADJ\tпонять_V...,Animacy=Inan|Case=Nom|Gender=Masc|Number=Sing\...
4,NORMAL,а когда мы статус агрогородка получили?,39,6,5.500000,True,False,False,False,False,False,0,0,0,False,False,статус\tагрогородка\tполучили,статус_NOUN\tагрогородка_NOUN\tполучить_VERB,Animacy=Inan|Case=Acc|Gender=Masc|Number=Sing\...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248276,NORMAL,правильно всё по пять (5)...,28,4,4.500000,True,False,False,False,False,False,0,0,0,False,True,правильно\tвсё\tпять,правильно_ADV\tвесь_DET\tпять_NUM,Degree=Pos\tCase=Nom|Gender=Neut|Number=Sing|P...
248277,INSULT,ёбанные нубы заходите на сервер мой ник _creep...,143,23,5.260870,False,True,False,False,False,False,2,0,3,False,False,ёбанные\tнубы\tзаходите\tсервер\tник\t_creepro...,ебанный_ADJ\tнуб_NOUN\tзаходить_VERB\tсервер_N...,Case=Nom|Degree=Pos|Number=Plur\tAnimacy=Inan|...
248278,NORMAL,а у меня наверное рекорд в 1962 году в училище...,188,30,4.833333,True,False,False,False,False,False,0,1,3,False,True,наверное\tрекорд\tгоду\tучилище\tкоренной\tзуб...,наверное_ADV\tрекорд_NOUN\tгод_NOUN\tучилище_N...,Degree=Pos\tAnimacy=Inan|Case=Nom|Gender=Masc|...
248279,NORMAL,спасибо всем большое),21,3,6.000000,True,False,False,False,False,False,0,0,2,False,False,спасибо\tвсем\tбольшое,спасибо_NOUN\tвсе_PRON\tбольшой_ADJ,Animacy=Inan|Case=Nom|Gender=Neut|Number=Sing\...
